In [1]:
123

123

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INP = "./data/steam_reviews_last180d.csv"
OUT = "./data/steam_reviews_random20k.csv"

N = 20_000
SEED = 42
CHUNKSIZE = 50_000  # 메모리 여유 있으면 500_000도 OK

rng = np.random.default_rng(SEED)

reservoir = None
seen = 0

for chunk in pd.read_csv(INP, chunksize=CHUNKSIZE, low_memory=False):
    m = len(chunk)
    if m == 0:
        continue

    # reservoir 채우기
    if reservoir is None:
        if m >= N:
            reservoir = chunk.sample(n=N, random_state=SEED).reset_index(drop=True)
            seen = m
            continue
        else:
            reservoir = chunk.copy().reset_index(drop=True)
            seen = m
            continue

    if len(reservoir) < N:
        need = N - len(reservoir)
        take = min(need, m)
        add = chunk.sample(n=take, random_state=SEED + seen).reset_index(drop=True)
        reservoir = pd.concat([reservoir, add], ignore_index=True)
        seen += m
        continue

    # reservoir sampling (균등 랜덤)
    r = rng.integers(0, seen + m, size=m)
    mask = r < N
    if mask.any():
        replace_idx = r[mask]
        replace_rows = chunk.loc[mask].to_numpy()
        reservoir.iloc[replace_idx] = replace_rows

    seen += m

if reservoir is None or len(reservoir) == 0:
    raise ValueError("입력 파일에서 데이터를 읽지 못했어. 경로/파일을 확인해줘.")

reservoir.to_csv(OUT, index=False)
print(f"✅ Saved {len(reservoir):,} rows -> {OUT}")


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd

INP = "./data/steam_reviews.csv"
OUT = "./data/steam_reviews_random50k.csv"

N = 20_000
SEED = 42
CHUNKSIZE = 100_000  # 메모리 여유 있으면 500_000도 OK

rng = np.random.default_rng(SEED)

reservoir = None
seen = 0

for chunk in pd.read_csv(INP, chunksize=CHUNKSIZE, low_memory=False):
    m = len(chunk)
    if m == 0:
        continue

    # reservoir 채우기
    if reservoir is None:
        if m >= N:
            reservoir = chunk.sample(n=N, random_state=SEED).reset_index(drop=True)
            seen = m
            continue
        else:
            reservoir = chunk.copy().reset_index(drop=True)
            seen = m
            continue

    if len(reservoir) < N:
        need = N - len(reservoir)
        take = min(need, m)
        add = chunk.sample(n=take, random_state=SEED + seen).reset_index(drop=True)
        reservoir = pd.concat([reservoir, add], ignore_index=True)
        seen += m
        continue

    # reservoir sampling (균등 랜덤)
    r = rng.integers(0, seen + m, size=m)
    mask = r < N
    if mask.any():
        replace_idx = r[mask]
        replace_rows = chunk.loc[mask].to_numpy()
        reservoir.iloc[replace_idx] = replace_rows

    seen += m

if reservoir is None or len(reservoir) == 0:
    raise ValueError("입력 파일에서 데이터를 읽지 못했어. 경로/파일을 확인해줘.")

reservoir.to_csv(OUT, index=False)
print(f"✅ Saved {len(reservoir):,} rows -> {OUT}")


✅ Saved 50,000 rows -> ./data/steam_reviews_random50k.csv


In [5]:
import numpy as np
import pandas as pd


In [6]:
temp_df = pd.read_csv('./data/steam_reviews_random20k.csv')
df = pd.read_csv('./data/steam_reviews_random15k.csv')
check_df = pd.read_csv('./data/games_march2025_cleaned.csv')

In [10]:
import duckdb
import pandas as pd

REVIEWS_BIG = "./data/steam_reviews_last180d.csv"      # 너희 원본 180만 파일 경로로 변경
GAMES_PATH  = "./data/games_march2025_cleaned.csv"
OUT_ALL     = "./data/reviews_joined_all_matched.csv"     # 매칭되는 전체 저장
OUT_20K     = "./data/reviews_joined_sample20k.csv"       # 그 중 2만 샘플

# 1) 원본 리뷰 파일에서 appid 컬럼명이 뭔지 자동 감지 (appid / app_id 둘 다 대응)
cols = pd.read_csv(REVIEWS_BIG, nrows=0).columns.str.lower().tolist()
if "appid" in cols:
    review_appid_col = "appid"
elif "app_id" in cols:
    review_appid_col = "app_id"
else:
    raise ValueError(f"리뷰 파일에서 appid 컬럼을 못 찾았어. 현재 컬럼 일부: {cols[:30]}")

con = duckdb.connect(database=":memory:")

# 2) games는 appid 중복이 있을 수 있으니 appid 기준 1개만 남기기
#    (name/genres만 쓰기)
con.execute(f"""
CREATE OR REPLACE TEMP VIEW games_dedup AS
SELECT appid, name, genres
FROM (
    SELECT
        appid,
        name,
        genres,
        ROW_NUMBER() OVER (PARTITION BY appid ORDER BY appid) AS rn
    FROM read_csv_auto('{GAMES_PATH}', header=True)
)
WHERE rn = 1;
""")

# 3) 리뷰 원본과 games를 INNER JOIN → 매칭되는 행만 남으니 game_name/genre 결측 0
#    (주의: 리뷰 파일 컬럼명이 appid가 아니라 app_id면 alias로 통일)
con.execute(f"""
CREATE OR REPLACE TEMP VIEW reviews_src AS
SELECT *, {review_appid_col} AS appid
FROM read_csv_auto('{REVIEWS_BIG}', header=True);
""")

# 4) 전체 매칭 데이터 저장
con.execute(f"""
COPY (
    SELECT
        r.*,
        g.name   AS game_name,
        g.genres AS genre
    FROM reviews_src r
    INNER JOIN games_dedup g
    ON r.appid = g.appid
) TO '{OUT_ALL}' (HEADER, DELIMITER ',');
""")

# 5) 그 결과에서 2만개 랜덤 샘플 저장 (재현 가능하게 seed 사용)
con.execute(f"""
COPY (
    SELECT *
    FROM (
        SELECT *, RANDOM() AS rnd
        FROM read_csv_auto('{OUT_ALL}', header=True)
    )
    ORDER BY rnd
    LIMIT 20000
) TO '{OUT_20K}' (HEADER, DELIMITER ',');
""")

# 6) 확인 출력
matched_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{OUT_ALL}', header=True)").fetchone()[0]
print("매칭된 전체 행 수:", matched_count)
print("저장 완료:", OUT_ALL)
print("샘플 20k 저장 완료:", OUT_20K)


매칭된 전체 행 수: 1030656
저장 완료: ./data/reviews_joined_all_matched.csv
샘플 20k 저장 완료: ./data/reviews_joined_sample20k.csv


In [7]:
import pandas as pd

REVIEWS_PATH = "./data/steam_reviews_random20k.csv"
GAMES_PATH   = "./data/games_march2025_cleaned.csv"
OUT_PATH     = "./data/steam_reviews_20k_with_gameinfo.csv"

# 1) 리뷰 20k 로드 (가볍기 때문에 통째로 OK)
reviews = pd.read_csv(REVIEWS_PATH, low_memory=False)

# appid 컬럼 체크
if "appid" not in reviews.columns:
    raise ValueError("리뷰 파일에 'appid' 컬럼이 없습니다. 컬럼명을 확인해주세요.")

# 2) 리뷰에 있는 appid만 뽑기
appid_set = set(reviews["appid"].dropna().astype("int64").unique())

# 3) games 파일에서 필요한 컬럼만 chunks로 읽어서 appid 매칭되는 것만 모으기
usecols = ["appid", "name", "genres"]
chunksize = 200_000  # PC 스펙에 맞춰 50k~500k 사이로 조절 가능

picked = []
for chunk in pd.read_csv(GAMES_PATH, usecols=usecols, chunksize=chunksize, low_memory=False):
    # appid 타입 맞추기 (문자열로 들어오는 경우 대비)
    chunk["appid"] = pd.to_numeric(chunk["appid"], errors="coerce")
    chunk = chunk.dropna(subset=["appid"])
    chunk["appid"] = chunk["appid"].astype("int64")

    # 리뷰 appid만 필터링
    chunk = chunk[chunk["appid"].isin(appid_set)]
    if not chunk.empty:
        picked.append(chunk)

# 매칭 결과가 하나도 없으면 여기서 바로 알 수 있게
if not picked:
    raise ValueError("games 파일에서 리뷰 appid와 매칭되는 행을 찾지 못했습니다. (appid 타입/값 확인 필요)")

games_small = pd.concat(picked, ignore_index=True)

# 4) appid 중복이 있을 수 있으니 1개로 정리 (첫 번째 값 사용)
games_small = games_small.drop_duplicates(subset=["appid"], keep="first")

# 5) 컬럼명 변경: name -> game_name, genres -> genre
games_small = games_small.rename(columns={"name": "game_name", "genres": "genre"})

# 6) 리뷰에 left merge (리뷰 20k 행은 유지, 게임정보가 없으면 NaN)
final = reviews.merge(games_small[["appid", "game_name", "genre"]], on="appid", how="left")

# 7) 저장
final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("완료! 저장 파일:", OUT_PATH)
print("리뷰 행 수:", len(reviews), "-> 결과 행 수:", len(final))
print("game_name 결측 비율:", final["game_name"].isna().mean())
print("genre 결측 비율:", final["genre"].isna().mean())

# (선택) 결과 5개만 확인
print(final[["appid", "game_name", "genre"]].head())


완료! 저장 파일: ./data/steam_reviews_20k_with_gameinfo.csv
리뷰 행 수: 20000 -> 결과 행 수: 20000
game_name 결측 비율: 0.4616
genre 결측 비율: 0.4616
     appid        game_name                             genre
0  2592160              NaN                               NaN
1  1086940  Baldur's Gate 3  ['Adventure', 'RPG', 'Strategy']
2  3527290              NaN                               NaN
3  2807960              NaN                               NaN
4  1091500   Cyberpunk 2077                           ['RPG']


In [42]:
pd.set_option('display.max_columns', None)

In [43]:
temp_df

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck
0,2592160,210566386,76561199159438195,121,20,136.0,0.0,128.0,NaN,1.764958e+09,english,"This game is a masterpiece, the story and anim...",1764089804,1764089804,True,0,0,0.500000,0,False,False,False,NaN,NaN,False
1,1086940,213893509,76561199153221721,210,5,23497.0,7233.0,16889.0,NaN,1.767327e+09,schinese,最近连续坏了我两个荣誉档，刚才推到最后马上决战主脑拿金骰子了给我卡主没法互动了，重新读档卡8...,1766387073,1766387073,False,2,3,0.509604,0,True,False,False,NaN,NaN,False
2,3527290,200007796,76561199081109350,49,18,1039.0,69.0,900.0,NaN,1.766834e+09,russian,ДА,1752756389,1752756389,True,1,0,0.523810,0,False,False,False,NaN,NaN,False
3,2807960,206731489,76561197998618809,0,11,5452.0,0.0,5223.0,NaN,1.762877e+09,english,"Do not buy. Bots, cheaters, XP farms, pointles...",1760467125,1762282524,False,0,0,0.500000,0,True,False,False,NaN,NaN,False
4,1091500,204037286,76561198269624998,145,25,29163.0,21.0,9578.0,NaN,1.766441e+09,russian,"Товарищи, это прекрасная игра! Хоть и игра не ...",1757533210,1757769883,True,0,0,0.500000,0,False,False,False,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,1888160,202679127,76561198138683802,338,27,1120.0,0.0,566.0,NaN,1.757370e+09,english,wow this is just like all the smut i've been r...,1755903367,1755903367,True,2,0,0.526102,1,False,False,False,NaN,NaN,False
19996,1808500,214584387,76561198159583458,0,12,4095.0,2984.0,2572.0,NaN,1.767588e+09,english,Just discovered this game about 2 weeks ago. V...,1767045400,1767045400,True,0,0,0.500000,0,True,False,False,NaN,NaN,False
19997,1091500,206395361,76561198833736707,30,2,6376.0,276.0,4102.0,NaN,1.766193e+09,brazilian,meu filho vai se chamar Jack Wells,1760145988,1760145988,True,0,0,0.500000,0,True,False,False,NaN,NaN,False
19998,381210,206314101,76561199630045696,0,3,5801.0,680.0,2242.0,NaN,1.766786e+09,vietnamese,:),1760075941,1760075941,True,0,0,0.500000,0,True,False,False,NaN,NaN,False


In [45]:
temp_df.isna().sum()

appid                              0
recommendationid                   0
steamid                            0
num_games_owned                    0
num_reviews_author                 0
playtime_forever                   0
playtime_last_two_weeks            0
playtime_at_review                 7
deck_playtime_at_review        19622
last_played                        0
language                           0
review                            76
timestamp_created                  0
timestamp_updated                  0
voted_up                           0
votes_up                           0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
developer_response             19970
timestamp_dev_responded        19970
primarily_steam_deck               0
dtype: int64

In [7]:
df

,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,17985344,644930,They Are Billions,66899072,schinese,666,1586361984,1586361984,True,0,...,True,False,False,76561198237086364,31,6,642.0,0.0,194.0,1.591716e+09
1,15288688,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,34432737,english,21/10 would play again but f$ck the hackers an...,1503476147,1510637434,True,0,...,False,False,True,76561198340731525,21,2,7467.0,0.0,2639.0,1.566988e+09
2,20773381,671510,Desolate,49320648,russian,"Есть баги, причем жесткие, у меня например не ...",1551552086,1551552086,True,0,...,True,False,False,76561198379377765,94,8,1191.0,0.0,1191.0,1.552250e+09
3,18067011,322330,Don't Starve Together,72177433,tchinese,←MOD愛好者（沒錯就是菜了）\n認真玩不加MOD會很有挑戰性\n加各種MOD的話就會是“相...,1594057760,1594057760,True,0,...,False,False,False,76561198997362162,21,9,4933.0,0.0,3257.0,1.609804e+09
4,18784582,105600,Terraria,50790126,russian,Просто супер . Игра класс . Геймплей тОп,1558643438,1558643438,True,0,...,True,False,False,76561198938103721,2,2,82.0,0.0,24.0,1.584604e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,15024934,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,37487733,russian,"Неплохой battleroyal, начиналось все даже хоро...",1511601263,1511601263,False,1,...,False,False,True,76561198028836378,62,3,63833.0,350.0,8314.0,1.610654e+09
14996,20639592,619080,SOS,40583361,schinese,我想说为什么我登陆不了这个什么鬼东西，服务器好低级,1520247645,1520247645,False,1,...,True,False,True,76561198820242872,1,1,34.0,0.0,34.0,1.520304e+09
14997,8092078,812140,Assassin's Creed Odyssey,73391756,brazilian,otimo jogo excelente,1595895148,1595895148,True,0,...,True,True,False,76561198855681844,30,15,1077.0,0.0,297.0,1.609277e+09
14998,6437666,359550,Tom Clancy's Rainbow Six Siege,47367554,russian,"Если преодолеть порог вхождения, то игра начне...",1544212201,1544683553,True,0,...,False,False,False,76561198039336315,178,14,87219.0,226.0,6513.0,1.610448e+09


app_id: 게임(앱)의 고유 ID (Steam의 앱 식별자)

app_name: 게임 이름

review_id: 리뷰의 고유 ID (리뷰 한 건을 구분하는 키)

language: 리뷰가 작성된 언어 코드/이름(예: en, ko 등)

review: 리뷰 본문 텍스트(사용자가 작성한 내용)

timestamp_created: 리뷰가 처음 작성된 시각(유닉스 타임스탬프일 가능성 높음: 초 단위)

timestamp_updated: 리뷰가 마지막으로 수정된 시각(작성 후 편집/업데이트된 마지막 시각)

recommended: 추천 여부(일반적으로 True/False)

True면 “이 게임 추천”, False면 “비추천”에 가까움

votes_helpful: 다른 유저들이 이 리뷰를 “도움됨(Helpful)”으로 누른 수

votes_funny: 다른 유저들이 이 리뷰를 “웃김(Funny)”으로 누른 수

weighted_vote_score: 리뷰의 투표 점수를 가중치로 환산한 점수(도움됨 투표 등의 영향이 반영된 “종합 점수” 느낌)

comment_count: 이 리뷰에 달린 댓글 수

steam_purchase: 이 게임을 Steam에서 구매했는지 여부

True면 Steam 구매, False면 Steam 외부 키/다른 경로일 수 있음(데이터 정의에 따라 다름)

received_for_free: 무료로 받은 게임인지 여부

이벤트/프로모션/리뷰키/선물 등으로 무료 획득했는지

written_during_early_access: 얼리 액세스(Early Access) 기간에 작성된 리뷰인지 여부

author.steamid: 리뷰 작성자(유저)의 Steam 고유 ID

author.num_games_owned: 작성자가 보유한 Steam 게임 수(라이브러리 규모)

author.num_reviews: 작성자가 Steam에 남긴 리뷰 개수(리뷰러 성향/활동성 지표)

author.playtime_forever: 해당 게임의 누적 플레이타임(보통 분 단위인 경우가 많음)

author.playtime_last_two_weeks: 최근 2주간 해당 게임 플레이타임(보통 분 단위)

author.playtime_at_review: 리뷰 작성 시점의 해당 게임 플레이타임(보통 분 단위)

author.last_played: 작성자가 해당 게임을 마지막으로 플레이한 시각(유닉스 타임스탬프일 가능성 높음)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      15000 non-null  int64  
 1   app_id                          15000 non-null  int64  
 2   app_name                        15000 non-null  object 
 3   review_id                       15000 non-null  int64  
 4   language                        15000 non-null  object 
 5   review                          14978 non-null  object 
 6   timestamp_created               15000 non-null  int64  
 7   timestamp_updated               15000 non-null  int64  
 8   recommended                     15000 non-null  bool   
 9   votes_helpful                   15000 non-null  int64  
 10  votes_funny                     15000 non-null  int64  
 11  weighted_vote_score             15000 non-null  float64
 12  comment_count                   

In [9]:
df.isna().sum()

Unnamed: 0                         0
app_id                             0
app_name                           0
review_id                          0
language                           0
review                            22
timestamp_created                  0
timestamp_updated                  0
recommended                        0
votes_helpful                      0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
author.steamid                     0
author.num_games_owned             0
author.num_reviews                 0
author.playtime_forever            0
author.playtime_last_two_weeks     0
author.playtime_at_review         14
author.last_played                 0
dtype: int64

In [13]:
df['is_englished'] = df['language'].astype(str).str.lower().eq('english').astype(int)



In [16]:
df[df['is_englished'] == 1].head()


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played,is_englished
1,15288688,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,34432737,english,21/10 would play again but f$ck the hackers an...,1503476147,1510637434,True,0,...,False,True,76561198340731525,21,2,7467.0,0.0,2639.0,1.566988e+09,1
9,11935015,1057090,Ori and the Will of the Wisps,65518431,english,"Once again, Ori finds a way to outshine many o...",1584843214,1584843214,True,0,...,False,False,76561198068723028,262,22,1014.0,0.0,1014.0,1.584841e+09,1
15,13930052,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,72281486,english,"fun game, fixed but still some cheaters",1594193966,1594193966,True,0,...,False,False,76561198378599529,12,2,14083.0,0.0,13763.0,1.605377e+09,1
17,2206485,289070,Sid Meier's Civilization VI,64125937,english,"It's good, but buy all the DLC to really get i...",1582684996,1582684996,True,0,...,False,False,76561198319129070,73,1,12833.0,0.0,11413.0,1.603768e+09,1
20,5626567,413150,Stardew Valley,68212977,english,This game is nothing short of captivating. The...,1588126449,1588126449,True,0,...,False,False,76561198995574800,3,1,15891.0,41.0,13993.0,1.610496e+09,1


영어가 아닌 행 삭제 + CSV로 새로 저장 (일회용)

In [15]:
df_en = df[df["is_englished"] == 1].copy()
df_en.reset_index(drop=True, inplace=True)

OUT_EN_ONLY = "./data/steam_reviews_en_only.csv"
df_en.to_csv(OUT_EN_ONLY, index=False, encoding="utf-8-sig")

print("Saved:", OUT_EN_ONLY, "Rows:", len(df_en))


Saved: ./data/steam_reviews_en_only.csv Rows: 6771


결측치 대체 없이 삭제

In [25]:
df = df.dropna(subset=['review', 'author.playtime_at_review']).copy()
df.reset_index(drop=True, inplace=True)


In [26]:
df.isna().sum()

Unnamed: 0                        0
app_id                            0
app_name                          0
review_id                         0
language                          0
review                            0
timestamp_created                 0
timestamp_updated                 0
recommended                       0
votes_helpful                     0
votes_funny                       0
weighted_vote_score               0
comment_count                     0
steam_purchase                    0
received_for_free                 0
written_during_early_access       0
author.steamid                    0
author.num_games_owned            0
author.num_reviews                0
author.playtime_forever           0
author.playtime_last_two_weeks    0
author.playtime_at_review         0
author.last_played                0
is_englished                      0
dtype: int64

df.describe()

In [31]:
import re
import pandas as pd

# 1) 긍정적인 단어 모음 사전
GOOD_PHRASES = [
    r"highly recommend(?:ed)?",
    r"definitely recommend",
    r"worth (?:buying|it|the money|the time)",
    r"great game",
    r"amazing game",
    r"awesome game",
    r"best game(?:s)?",
]

GOOD_WORDS = [
    r"awesome", r"amazing", r"great", r"excellent", r"fantastic", r"incredible",
    r"masterpiece", r"perfect", r"love", r"fun", r"enjoy", r"recommend", r"worth",
]

# 2) good 오탐 방지용 "부정 구문" (good 단어가 있어도 여기에 걸리면 good=0 처리)
NEGATE_PHRASES = [
    r"not\s+good",
    r"not\s+great",
    r"not\s+worth",
    r"(?:do\s*not|don't|dont)\s+recommend",
    r"(?:do\s*not|don't|dont)\s+buy",
    r"can't\s+recommend|cant\s+recommend",
]

def build_good_regex(phrases, words):
    parts = []
    parts += [f"(?:{p})" for p in phrases]
    parts += [fr"\b{w}\b" for w in words]  # 단어 경계로 오탐 감소
    return re.compile("|".join(parts), flags=re.IGNORECASE)

GOOD_RE = build_good_regex(GOOD_PHRASES, GOOD_WORDS)
NEG_RE  = re.compile("|".join([f"(?:{p})" for p in NEGATE_PHRASES]), flags=re.IGNORECASE)

def add_good_flag(df, text_col="review"):
    out = df.copy()
    text = out[text_col].fillna("").astype(str).str.lower()

    good_hit = text.str.contains(GOOD_RE, regex=True)
    neg_hit  = text.str.contains(NEG_RE,  regex=True)

    # good 단어/구문이 있어도, 명확한 부정구문이 있으면 good=0
    out["good_review"] = (good_hit & (~neg_hit)).astype(int)
    return out


In [32]:
# df에 적용
df_good = add_good_flag(df)

df_good["good_review"].value_counts()

good_review
0    11792
1     3172
Name: count, dtype: int64

In [33]:
# df_en (영어-only 데이터프레임)에 적용
df_good = add_good_flag(df_en)

df_good["good_review"].value_counts()


good_review
0    3811
1    2960
Name: count, dtype: int64

In [35]:
df_good_all = add_good_flag(df)

non_eng_good = df_good_all[
    (df_good_all["language"].astype(str).str.lower() != "english") &
    (df_good_all["good_review"] == 1)
]

print("비영어인데 good=1 개수:", len(non_eng_good))
non_eng_good["language"].value_counts().head(10)


비영어인데 good=1 개수: 214


language
french       43
german       31
swedish      15
dutch        15
polish       14
schinese     12
brazilian    12
russian      10
finnish       8
czech         7
Name: count, dtype: int64

In [11]:
sample_df = pd.read_csv('./data/reviews_joined_sample20k.csv')

In [12]:
sample_df

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,...,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre,rnd
0,236390,214935645,76561199408691293,70,8,6553,199,6528,NaN,1767374825,...,False,False,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulatio...",6.626704e-07
1,381210,210513031,76561199034590636,0,9,23353,1308,16597,NaN,1767470513,...,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],2.654878e-06
2,1551360,205317346,76561199062613729,71,2,4780,0,4224,NaN,1761669035,...,True,False,False,NaN,NaN,False,1551360,Forza Horizon 5,"['Action', 'Adventure', 'Racing', 'Simulation'...",3.815672e-06
3,381210,201395452,76561198998550758,7,2,52724,119,41315,NaN,1766683805,...,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],5.021407e-06
4,236390,205324525,76561199588258066,0,1,604,0,218,NaN,1766288019,...,False,True,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulatio...",5.237950e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2139460,214989253,76561199805450933,0,1,5404,5404,4578,NaN,1767509031,...,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', '...",1.913930e-02
19996,413150,207624891,76561197964370149,0,1,17457,0,904,NaN,1765374033,...,True,False,False,NaN,NaN,False,413150,Stardew Valley,"['Indie', 'RPG', 'Simulation']",1.913939e-02
19997,413150,206013726,76561199118992449,0,1,11876,0,1526,NaN,1764769196,...,True,False,False,NaN,NaN,False,413150,Stardew Valley,"['Indie', 'RPG', 'Simulation']",1.913994e-02
19998,2651280,199703559,76561198393336271,533,25,2191,0,2190,NaN,1752409648,...,True,False,False,NaN,NaN,False,2651280,Marvel's Spider-Man 2,"['Action', 'Adventure']",1.914117e-02


In [37]:
review_dt = pd.to_datetime(sample_df["timestamp_created"], unit="s")
last_dt   = pd.to_datetime(sample_df["last_played"], unit="s")

sample_df["churn_90d"] = (last_dt <= (review_dt + pd.Timedelta(days=1))).astype(int)

In [38]:
sample_df['churn_90d'].value_counts()

churn_90d
0    14673
1     5327
Name: count, dtype: int64

In [54]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/reviews_joined_sample20k.csv", low_memory=False)

review_dt = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
last_dt   = pd.to_datetime(df["last_played"], unit="s", errors="coerce")

# 스냅샷 시점(데이터에서 가장 최근 리뷰 시점) - 필요하면 last_dt.max()로 바꿔도 됨
snap_dt = review_dt.max()

# 마지막 플레이로부터 경과 일수(= 비활성 기간)
days_inactive = (snap_dt - last_dt).dt.days
print("days_inactive")
print(days_inactive)
bins = [-1, 30, 60, 90, np.inf]
labels = [0, 1, 2, 3]

df["churn_stage"] = pd.cut(days_inactive, bins=bins, labels=labels).astype("Int64")
stage_name = {0: "Active", 1: "At-risk", 2: "Churned", 3: "Lost"}
df["churn_stage_name"] = df["churn_stage"].map(stage_name)

print(df["churn_stage"].value_counts(dropna=False).sort_index())
print(days_inactive.describe())


days_inactive
0          3
1          2
2         69
3         11
4         15
        ... 
19995      1
19996     26
19997     33
19998    176
19999      6
Name: last_played, Length: 20000, dtype: int64
churn_stage
0       10882
1        3262
2        1588
3        4025
<NA>      243
Name: count, dtype: Int64
count    20000.00000
mean        59.17740
std        132.97787
min         -1.00000
25%          4.00000
50%         22.00000
75%         71.00000
max       3446.00000
Name: last_played, dtype: float64


In [55]:
sample_df.isna().sum()

appid                              0
recommendationid                   0
steamid                            0
num_games_owned                    0
num_reviews_author                 0
playtime_forever                   0
playtime_last_two_weeks            0
playtime_at_review                 0
deck_playtime_at_review        19622
last_played                        0
language                           0
review                            65
timestamp_created                  0
timestamp_updated                  0
voted_up                           0
votes_up                           0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
developer_response             19944
timestamp_dev_responded        19944
primarily_steam_deck               0
appid_1                            0
game_name                          0
g

In [51]:
IN_PATH = "./data/reviews_joined_sample20k.csv"
df = pd.read_csv(IN_PATH, low_memory=False)

review_dt = pd.to_datetime(df["timestamp_created"], unit="s")
last_dt = pd.to_datetime(df["last_played"], unit="s")

# 리뷰 시점 기준, 마지막 플레이 이후 경과 일수
df["days_since_last_play"] = (last_dt - review_dt).dt.days
bins = [-float("inf"), 14, 45, 90, float("inf")]
labels = [0, 1, 2, 3]

df["stage"] = pd.cut(
    df["days_since_last_play"],
    bins=bins,
    labels=labels
).astype(int)

print(df['stage'].value_counts())

stage
0    9488
1    4370
3    3326
2    2816
Name: count, dtype: int64


In [58]:
print(sample_df['genre'].unique())

["['Action', 'Massively Multiplayer', 'Simulation', 'Free To Play']"
 "['Action']" "['Action', 'Adventure', 'Racing', 'Simulation', 'Sports']"
 "['Action', 'Adventure', 'Massively Multiplayer', 'Free To Play']"
 "['Action', 'Strategy']" "['Action', 'Early Access']"
 "['Indie', 'RPG', 'Simulation']" "['Adventure', 'Indie', 'Simulation']"
 "['Action', 'Adventure', 'RPG']" "['Casual', 'Simulation', 'Strategy']"
 "['Action', 'Adventure']"
 "['Adventure', 'Casual', 'Simulation', 'Free To Play']"
 "['Adventure', 'Indie', 'Simulation', 'Strategy']"
 "['Action', 'Adventure', 'RPG', 'Early Access']" "['RPG']"
 "['Action', 'Adventure', 'Indie', 'Early Access']"
 "['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
 "['Action', 'Free To Play']" "['Simulation', 'Strategy', 'Free To Play']"
 "['Action', 'Adventure', 'Free To Play']"
 "['Action', 'Indie', 'RPG', 'Simulation', 'Strategy']"
 "['Adventure', 'RPG', 'Strategy']"
 "['Action', 'Adventure', 'Indie', 'Simulation']" "['A

In [43]:
import pandas as pd
import numpy as np

# 파일명
IN_PATH = "./data/reviews_joined_sample20k.csv"

df = pd.read_csv(IN_PATH, low_memory=False)

# 컬럼명 (너희 파일 기준)
review_col = "timestamp_created"
last_col = "last_played"

# Unix timestamp(초) -> datetime
review_dt = pd.to_datetime(df[review_col], unit="s", errors="coerce")
last_dt   = pd.to_datetime(df[last_col], unit="s", errors="coerce")

# 리뷰 이후 마지막 플레이까지 경과 '일수'
days_gap = (last_dt - review_dt).dt.days

# (선택) 음수(리뷰보다 과거 last_played)는 0으로 보정하고 싶으면 주석 해제
# days_gap = days_gap.clip(lower=0)

# Stage 구간화: 0–14, 15–45, 46–90, 91+
bins = [-1, 14, 45, 90, np.inf]
labels = [0, 1, 2, 3]
df["churn_stage"] = pd.cut(days_gap, bins=bins, labels=labels).astype("Int64")

# 상태명 매핑
stage_name = {0: "Active", 1: "At-risk", 2: "Churned", 3: "Lost"}
df["churn_stage_name"] = df["churn_stage"].map(stage_name)

# 확인 출력
print("=== churn_stage distribution ===")
print(df["churn_stage"].value_counts(dropna=False).sort_index())

print("\n=== churn_stage_name distribution ===")
print(df["churn_stage_name"].value_counts(dropna=False))

print("\n=== days_gap summary (days_since_review_to_last_played) ===")
print(days_gap.describe())


=== churn_stage distribution ===
churn_stage
0       5901
1       4370
2       2816
3       3326
<NA>    3587
Name: count, dtype: Int64

=== churn_stage_name distribution ===
churn_stage_name
Active     5901
At-risk    4370
NaN        3587
Lost       3326
Churned    2816
Name: count, dtype: int64

=== days_gap summary (days_since_review_to_last_played) ===
count    20000.000000
mean        18.324050
std        134.523575
min      -3439.000000
25%          0.000000
50%         18.000000
75%         64.000000
max        179.000000
dtype: float64


In [46]:
dist = df["churn_stage"].value_counts(normalize=True).sort_index()
print(dist)


churn_stage
0    0.359532
1    0.266252
2    0.171571
3    0.202644
Name: proportion, dtype: Float64


In [39]:
review_dt = pd.to_datetime(sample_df["timestamp_created"], unit="s")
last_dt   = pd.to_datetime(sample_df["last_played"], unit="s")

# 1) 기존 방식
churn_90_last = (last_dt <= review_dt + pd.Timedelta(days=90)).astype(int)

# 2) 최근 180일 리뷰만 + 기존 방식
end_dt = review_dt.max()
mask_recent = review_dt >= (end_dt - pd.Timedelta(days=180))
churn_90_last_recent = (last_dt[mask_recent] <= review_dt[mask_recent] + pd.Timedelta(days=90)).astype(int)

# 3) 플레이타임 증가량 기반
delta = sample_df["playtime_forever"] - sample_df["playtime_at_review"]
churn_playtime = (delta < 60).astype(int)

print("기존 last_played 90d 이탈비율:", churn_90_last.mean())
print("최근180일만 last_played 90d 이탈비율:", churn_90_last_recent.mean())
print("playtime delta<60 이탈비율:", churn_playtime.mean())


기존 last_played 90d 이탈비율: 0.8306
최근180일만 last_played 90d 이탈비율: 0.8306
playtime delta<60 이탈비율: 0.2878


In [40]:
((last_dt - review_dt).dt.days.describe())

count    20000.000000
mean        18.324050
std        134.523575
min      -3439.000000
25%          0.000000
50%         18.000000
75%         64.000000
max        179.000000
dtype: float64